# L4S_01 — Modelos Clásicos (RF · SVM · LR) — 5-Fold CV · Dataset Completo
**Proyecto:** Detección de Deslizamientos — Landslide4Sense  
**Protocolo:** 5-Fold Stratified CV · 3 799 parches completos  
**Comparable con:** Ghorbanzadeh et al. (2022), Youssef & Pourghasemi (2021)

> **Diferencias respecto a la versión restringida (notebook 03):**  
> • `N_SAMPLES = 3 799` (antes 1 500) — dataset completo sin submuestreo  
> • `n_splits = 5` — validación cruzada estándar de la literatura  
> • AUC-PR incluida — métrica adicional recomendada para clases desbalanceadas  
> • Tabla de comparación con resultados publicados al final del notebook

In [ ]:
# ── Celda 1: Dependencias, Drive y carga de datos ──────────────────────────
from google.colab import drive
import h5py, json, warnings, time
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm.auto import tqdm
from skimage.feature import hog

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    f1_score, precision_score, recall_score,
    roc_auc_score, jaccard_score,
    average_precision_score,
    confusion_matrix, ConfusionMatrixDisplay,
    roc_curve, precision_recall_curve, auc
)

warnings.filterwarnings('ignore')

drive.mount('/content/drive')

base_path = Path('/content/drive/MyDrive/Landslide4Sense')
img_dirs  = list(base_path.glob('**/TrainData/img'))

if img_dirs:
    train_img_dir  = img_dirs[0]
    train_mask_dir = train_img_dir.parent / 'mask'
    img_list  = sorted(list(train_img_dir.glob('*.h5')))
    # Corrección naming: image_N.h5 → mask_N.h5
    mask_list = [train_mask_dir / p.name.replace('image_', 'mask_') for p in img_list]
    print(f'✅ {len(img_list)} parches detectados')
    print(f'   Positivos: (se calculará al extraer features)')
else:
    raise FileNotFoundError('❌ No se encontró TrainData. Verifica la ruta en Drive.')


## 1. Extracción de características — Dataset completo (3 799 parches)

Vector por parche: HOG(RGB S2) + pendiente DEM + NDVI + SAR-VH = **1 767 dimensiones**

> **Nota de tiempo:** ~8–12 min en Colab para 3 799 parches.  
> Si el entorno expira, el caché JSON en Drive permite reanudar desde la celda 2.

In [ ]:
# ── Celda 2: Extracción de características (ALL samples) ───────────────────
#
# CAMBIO CLAVE vs notebook 03: N_SAMPLES = len(img_list) → 3 799 parches
# Sin submuestreo → más representativo → comparable con la literatura

CACHE_PATH = base_path / 'results' / 'features_cache_full.npz'
CACHE_PATH.parent.mkdir(parents=True, exist_ok=True)

def extract_features(img_path, mask_path):
    """HOG sobre RGB falso color + pendiente DEM + NDVI + SAR-VH."""
    with h5py.File(img_path, 'r') as hf:
        key = list(hf.keys())[0]
        patch = hf[key][()].astype(np.float32)   # (128, 128, 14)
    with h5py.File(mask_path, 'r') as hf:
        key = list(hf.keys())[0]
        mask = hf[key][()].astype(np.float32)    # (128, 128)

    # Normalizar por canal para HOG
    rgb = patch[:, :, [3, 2, 1]]   # B4-B3-B2 (R-G-B Sentinel-2)
    for c in range(3):
        ch = rgb[:, :, c]
        rng = ch.max() - ch.min() + 1e-8
        rgb[:, :, c] = (ch - ch.min()) / rng

    hog_feat = hog(rgb, orientations=9, pixels_per_cell=(8,8),
                   cells_per_block=(2,2), channel_axis=-1)  # (1764,)

    slope  = float(patch[:, :, 10].mean())   # DEM pendiente
    b8     = patch[:, :, 3].astype(float)    # NIR (B8 S2)
    b4     = patch[:, :, 2].astype(float)    # Rojo
    ndvi   = float(((b8 - b4) / (b8 + b4 + 1e-8)).mean())
    sar_vh = float(patch[:, :, 8].mean())    # SAR VH

    features = np.concatenate([hog_feat, [slope, ndvi, sar_vh]])  # (1767,)
    label    = int(mask.max() > 0)
    return features, label

if CACHE_PATH.exists():
    cache = np.load(CACHE_PATH)
    X, y = cache['X'], cache['y']
    print(f'✅ Caché cargado: {X.shape[0]} muestras, {X.shape[1]} features')
else:
    print(f'Extrayendo features de {len(img_list)} parches...')
    t0 = time.time()
    X, y = [], []
    for img_p, mask_p in tqdm(zip(img_list, mask_list), total=len(img_list)):
        feat, label = extract_features(img_p, mask_p)
        X.append(feat)
        y.append(label)
    X = np.array(X, dtype=np.float32)
    y = np.array(y, dtype=np.int32)
    np.savez_compressed(CACHE_PATH, X=X, y=y)
    print(f'✅ Features extraídas en {(time.time()-t0)/60:.1f} min → caché guardado')

print(f'Dataset: {X.shape} | Positivos: {y.sum()} ({y.mean():.2%}) | Negativos: {(1-y).sum()}')


## 2. Validación cruzada 5-Fold — Protocolo estándar literatura

`n_splits=5` con estratificación por etiqueta.  
Misma semilla `random_state=42` que todos los notebooks del proyecto.

In [ ]:
# ── Celda 3: Configuración K-Fold ──────────────────────────────────────────
N_FOLDS = 5
RANDOM_STATE = 42
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
print(f'Protocolo: {N_FOLDS}-Fold Stratified CV | N={len(X)} | seed={RANDOM_STATE}')
print(f'Tamaño aprox. por fold → train: {int(len(X)*0.8)} | val: {int(len(X)*0.2)}')


## 3. Modelo 1 — Logistic Regression

In [ ]:
# ── Celda 4: Logistic Regression — 5-Fold ──────────────────────────────────
print('=' * 60)
print('  MODELO 1 — LOGISTIC REGRESSION (5-Fold CV)')
print('=' * 60)

lr_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(C=1.0, class_weight='balanced',
                               max_iter=1000, random_state=RANDOM_STATE))
])

lr_records = []
lr_probs_all, lr_true_all = [], []

for fold, (t_idx, v_idx) in enumerate(skf.split(X, y)):
    t0 = time.time()
    lr_pipeline.fit(X[t_idx], y[t_idx])
    preds = lr_pipeline.predict(X[v_idx])
    probs = lr_pipeline.predict_proba(X[v_idx])[:, 1]

    f1   = f1_score(y[v_idx], preds,  zero_division=0)
    prec = precision_score(y[v_idx], preds, zero_division=0)
    rec  = recall_score(y[v_idx],  preds,  zero_division=0)
    auc_roc = roc_auc_score(y[v_idx], probs)
    auc_pr  = average_precision_score(y[v_idx], probs)
    iou  = jaccard_score(y[v_idx], preds, zero_division=0)

    lr_records.append({'fold': fold+1, 'f1': f1, 'prec': prec, 'rec': rec,
                        'auc_roc': auc_roc, 'auc_pr': auc_pr, 'iou': iou})
    lr_probs_all.extend(probs); lr_true_all.extend(y[v_idx])
    print(f'  Fold {fold+1} | F1={f1:.4f} | AUC-ROC={auc_roc:.4f} | AUC-PR={auc_pr:.4f} '
          f'| IoU={iou:.4f} | {time.time()-t0:.1f}s')

lr_mean_f1   = np.mean([r['f1']      for r in lr_records])
lr_std_f1    = np.std( [r['f1']      for r in lr_records])
lr_mean_auc  = np.mean([r['auc_roc'] for r in lr_records])
lr_mean_aupr = np.mean([r['auc_pr']  for r in lr_records])
lr_mean_iou  = np.mean([r['iou']     for r in lr_records])
lr_mean_prec = np.mean([r['prec']    for r in lr_records])
lr_mean_rec  = np.mean([r['rec']     for r in lr_records])

print(f'\n  RESUMEN LR → F1={lr_mean_f1:.4f}±{lr_std_f1:.4f} | AUC-ROC={lr_mean_auc:.4f} | AUC-PR={lr_mean_aupr:.4f}')


## 4. Modelo 2 — SVM kernel RBF

In [ ]:
# ── Celda 5: SVM RBF — 5-Fold ───────────────────────────────────────────────
#
# ⚠️  TIEMPO ESTIMADO con 3 799 muestras: ~15–30 min en Colab CPU.
# SVC escala O(n²) en memoria → la Gram matrix es ~3799² × 8B ≈ 115 MB.
# Si el runtime expira, usa LinearSVC como alternativa rápida (ver comentario).
#
# Alternativa rápida (cambia estas líneas si el tiempo es crítico):
# from sklearn.svm import LinearSVC
# from sklearn.calibration import CalibratedClassifierCV
# svm_base = CalibratedClassifierCV(LinearSVC(C=1.0, class_weight='balanced',
#                                             max_iter=5000, random_state=42))
# svm_pipeline = Pipeline([('scaler', StandardScaler()), ('clf', svm_base)])

print('=' * 60)
print('  MODELO 2 — SVM kernel RBF (5-Fold CV)')
print('  ⚠️  Puede tomar 15-30 min con dataset completo')
print('=' * 60)

svm_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', SVC(kernel='rbf', C=1.0, gamma='scale',
                probability=True, class_weight='balanced', random_state=RANDOM_STATE))
])

svm_records = []
svm_probs_all, svm_true_all = [], []

for fold, (t_idx, v_idx) in enumerate(skf.split(X, y)):
    t0 = time.time()
    svm_pipeline.fit(X[t_idx], y[t_idx])
    preds = svm_pipeline.predict(X[v_idx])
    probs = svm_pipeline.predict_proba(X[v_idx])[:, 1]

    f1   = f1_score(y[v_idx], preds,  zero_division=0)
    prec = precision_score(y[v_idx], preds, zero_division=0)
    rec  = recall_score(y[v_idx],  preds,  zero_division=0)
    auc_roc = roc_auc_score(y[v_idx], probs)
    auc_pr  = average_precision_score(y[v_idx], probs)
    iou  = jaccard_score(y[v_idx], preds, zero_division=0)

    svm_records.append({'fold': fold+1, 'f1': f1, 'prec': prec, 'rec': rec,
                         'auc_roc': auc_roc, 'auc_pr': auc_pr, 'iou': iou})
    svm_probs_all.extend(probs); svm_true_all.extend(y[v_idx])
    print(f'  Fold {fold+1} | F1={f1:.4f} | AUC-ROC={auc_roc:.4f} | AUC-PR={auc_pr:.4f} '
          f'| IoU={iou:.4f} | {(time.time()-t0)/60:.1f}min')

svm_mean_f1   = np.mean([r['f1']      for r in svm_records])
svm_std_f1    = np.std( [r['f1']      for r in svm_records])
svm_mean_auc  = np.mean([r['auc_roc'] for r in svm_records])
svm_mean_aupr = np.mean([r['auc_pr']  for r in svm_records])
svm_mean_iou  = np.mean([r['iou']     for r in svm_records])
svm_mean_prec = np.mean([r['prec']    for r in svm_records])
svm_mean_rec  = np.mean([r['rec']     for r in svm_records])

print(f'\n  RESUMEN SVM → F1={svm_mean_f1:.4f}±{svm_std_f1:.4f} | AUC-ROC={svm_mean_auc:.4f} | AUC-PR={svm_mean_aupr:.4f}')


## 5. Modelo 3 — Random Forest

In [ ]:
# ── Celda 6: Random Forest — 5-Fold ────────────────────────────────────────
print('=' * 60)
print('  MODELO 3 — RANDOM FOREST (5-Fold CV)')
print('=' * 60)

rf_model = RandomForestClassifier(n_estimators=200, n_jobs=-1,
                                   class_weight='balanced', random_state=RANDOM_STATE)

rf_records = []
rf_probs_all, rf_true_all = [], []
rf_last_model, rf_last_val_idx = None, None

for fold, (t_idx, v_idx) in enumerate(skf.split(X, y)):
    t0 = time.time()
    rf_model.fit(X[t_idx], y[t_idx])
    preds = rf_model.predict(X[v_idx])
    probs = rf_model.predict_proba(X[v_idx])[:, 1]

    f1   = f1_score(y[v_idx], preds,  zero_division=0)
    prec = precision_score(y[v_idx], preds, zero_division=0)
    rec  = recall_score(y[v_idx],  preds,  zero_division=0)
    auc_roc = roc_auc_score(y[v_idx], probs)
    auc_pr  = average_precision_score(y[v_idx], probs)
    iou  = jaccard_score(y[v_idx], preds, zero_division=0)

    rf_records.append({'fold': fold+1, 'f1': f1, 'prec': prec, 'rec': rec,
                        'auc_roc': auc_roc, 'auc_pr': auc_pr, 'iou': iou})
    rf_probs_all.extend(probs); rf_true_all.extend(y[v_idx])
    rf_last_model, rf_last_val_idx = rf_model, v_idx
    print(f'  Fold {fold+1} | F1={f1:.4f} | AUC-ROC={auc_roc:.4f} | AUC-PR={auc_pr:.4f} '
          f'| IoU={iou:.4f} | {time.time()-t0:.1f}s')

rf_mean_f1   = np.mean([r['f1']      for r in rf_records])
rf_std_f1    = np.std( [r['f1']      for r in rf_records])
rf_mean_auc  = np.mean([r['auc_roc'] for r in rf_records])
rf_mean_aupr = np.mean([r['auc_pr']  for r in rf_records])
rf_mean_iou  = np.mean([r['iou']     for r in rf_records])
rf_mean_prec = np.mean([r['prec']    for r in rf_records])
rf_mean_rec  = np.mean([r['rec']     for r in rf_records])

print(f'\n  RESUMEN RF → F1={rf_mean_f1:.4f}±{rf_std_f1:.4f} | AUC-ROC={rf_mean_auc:.4f} | AUC-PR={rf_mean_aupr:.4f}')

# Importancia de features (último fold)
importances = rf_last_model.feature_importances_
top_idx = np.argsort(importances)[::-1][:15]
feature_names = [f'HOG_{i}' for i in range(1764)] + ['DEM_slope', 'NDVI', 'SAR_VH']
print('\nTop 15 features por importancia RF:')
for rank, idx in enumerate(top_idx, 1):
    print(f'  {rank:2d}. {feature_names[idx]:<20s} {importances[idx]:.5f}')


## 6. Tabla comparativa y visualizaciones

In [ ]:
# ── Celda 7: Tabla comparativa y curvas ROC/PR ──────────────────────────────
models_summary = [
    {'name': 'Logistic Regression', 'color': '#4e8df5',
     'f1': lr_mean_f1,   'std': lr_std_f1,   'auc': lr_mean_auc,   'aupr': lr_mean_aupr,
     'prec': lr_mean_prec, 'rec': lr_mean_rec, 'iou': lr_mean_iou,
     'probs': lr_probs_all, 'true': lr_true_all},
    {'name': 'SVM (RBF)',           'color': '#f5a623',
     'f1': svm_mean_f1,  'std': svm_std_f1,  'auc': svm_mean_auc,  'aupr': svm_mean_aupr,
     'prec': svm_mean_prec, 'rec': svm_mean_rec, 'iou': svm_mean_iou,
     'probs': svm_probs_all, 'true': svm_true_all},
    {'name': 'Random Forest',       'color': '#2ecc71',
     'f1': rf_mean_f1,   'std': rf_std_f1,   'auc': rf_mean_auc,   'aupr': rf_mean_aupr,
     'prec': rf_mean_prec, 'rec': rf_mean_rec, 'iou': rf_mean_iou,
     'probs': rf_probs_all, 'true': rf_true_all},
]

print('\n' + '='*75)
print(f'  {'Modelo':<22} {'F1±std':>12} {'AUC-ROC':>9} {'AUC-PR':>8} {'IoU':>7} {'Prec':>7} {'Rec':>7}')
print('='*75)
for m in models_summary:
    print(f"  {m['name']:<22} {m['f1']:.4f}±{m['std']:.4f} {m['auc']:>9.4f} {m['aupr']:>8.4f} {m['iou']:>7.4f} {m['prec']:>7.4f} {m['rec']:>7.4f}")
print('='*75)
print('  Protocolo: 5-Fold Stratified CV | N=3799 | seed=42')

# Figura comparativa
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Modelos Clásicos — 5-Fold CV — Dataset Completo (3 799 parches)', fontsize=13, fontweight='bold')

# F1 por fold
ax = axes[0]
fold_f1_data = [
    [r['f1'] for r in lr_records],
    [r['f1'] for r in svm_records],
    [r['f1'] for r in rf_records],
]
colors = [m['color'] for m in models_summary]
names  = [m['name']  for m in models_summary]
bp = ax.boxplot(fold_f1_data, patch_artist=True, labels=['LR','SVM','RF'])
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color); patch.set_alpha(0.7)
ax.set_ylabel('F1-score'); ax.set_title('F1 por Fold'); ax.set_ylim(0, 1); ax.grid(alpha=0.3)

# Curvas ROC
ax = axes[1]
for m in models_summary:
    fpr, tpr, _ = roc_curve(m['true'], m['probs'])
    ax.plot(fpr, tpr, color=m['color'], label=f"{m['name']} (AUC={m['auc']:.3f})", lw=2)
ax.plot([0,1],[0,1],'k--',lw=1)
ax.set_xlabel('FPR'); ax.set_ylabel('TPR'); ax.set_title('Curvas ROC'); ax.legend(fontsize=9); ax.grid(alpha=0.3)

# Curvas PR
ax = axes[2]
for m in models_summary:
    prec_curve, rec_curve, _ = precision_recall_curve(m['true'], m['probs'])
    ax.plot(rec_curve, prec_curve, color=m['color'], label=f"{m['name']} (AP={m['aupr']:.3f})", lw=2)
ax.set_xlabel('Recall'); ax.set_ylabel('Precision'); ax.set_title('Curvas PR'); ax.legend(fontsize=9); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(base_path / 'results' / 'L4S_01_classicos_5fold_comparison.png', dpi=150, bbox_inches='tight')
plt.show()


## 7. Comparación con la literatura

| Trabajo | Modelo | F1 | AUC | Dataset | N train | CV |
|---------|--------|-----|-----|---------|---------|-----|
| Ghorbanzadeh et al. 2022 | Ensemble ML | ~0.80 | ~0.88 | Landslide4Sense | 3 799 | Official split |
| Youssef & Pourghasemi 2021 | RF | 0.82–0.91 AUC | — | Asir, Saudi Arabia | — | — |
| **Este trabajo (5-Fold)** | RF (HOG+DEM+NDVI+SAR) | *ver arriba* | *ver arriba* | Landslide4Sense | 3 799 | 5-Fold |
| **Este trabajo (2-Fold)** | RF (HOG+DEM+NDVI+SAR) | 0.837 | 0.808 | Landslide4Sense | 1 500 | 2-Fold |

> La columna "Este trabajo (5-Fold)" se completa automáticamente con los resultados de las celdas anteriores.

In [ ]:
# ── Celda 8: Guardar resultados en Drive ───────────────────────────────────
out_dir = base_path / 'results' / 'comparable_literature' / 'classicos_5fold'
out_dir.mkdir(parents=True, exist_ok=True)

results = {
    'protocol': {'n_folds': 5, 'n_samples': len(X), 'seed': 42,
                 'features': 'HOG(9,8,2,RGB) + DEM_slope + NDVI + SAR_VH',
                 'comparable_with': 'Ghorbanzadeh2022, Youssef2021'},
    'models': {
        'logistic_regression': {
            'mean_f1': float(lr_mean_f1), 'std_f1': float(lr_std_f1),
            'mean_auc_roc': float(lr_mean_auc), 'mean_auc_pr': float(lr_mean_aupr),
            'mean_iou': float(lr_mean_iou), 'folds': lr_records
        },
        'svm_rbf': {
            'mean_f1': float(svm_mean_f1), 'std_f1': float(svm_std_f1),
            'mean_auc_roc': float(svm_mean_auc), 'mean_auc_pr': float(svm_mean_aupr),
            'mean_iou': float(svm_mean_iou), 'folds': svm_records
        },
        'random_forest': {
            'mean_f1': float(rf_mean_f1), 'std_f1': float(rf_std_f1),
            'mean_auc_roc': float(rf_mean_auc), 'mean_auc_pr': float(rf_mean_aupr),
            'mean_iou': float(rf_mean_iou), 'folds': rf_records
        }
    }
}

with open(out_dir / 'kfold5_summary.json', 'w') as f:
    json.dump(results, f, indent=2)

print(f'✅ Resultados guardados en: {out_dir}')
print(f'   → kfold5_summary.json')
print(f'   → L4S_01_classicos_5fold_comparison.png')
print('\n📌 Resumen final:')
for m in models_summary:
    print(f"   {m['name']:<22} F1={m['f1']:.4f}±{m['std']:.4f} | AUC-ROC={m['auc']:.4f} | AUC-PR={m['aupr']:.4f}")
